In [3]:
from typing_extensions import TypedDict, Literal, Annotated, Dict 
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.types import Command
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage
from pathlib import Path
import os
import json
import subprocess

In [4]:
TOOLS = [
    {"name": "bash",       "description": "Run a shell command."},
    {"name": "read_file",  "description": "Read file contents."},
    {"name": "write_file", "description": "Write content to file."},
    {"name": "edit_file",  "description": "Replace text in file once."},
    {"name": "glob",       "description": "Find files by pattern."},
]

SYSTEM = f"""You are a coding agent at {os.getcwd()}. Use the tools available to solve tasks. Act, don't explain. If a task has been completed, output STOP.
The following tools are available: {str(TOOLS)}

"""

MAX_ITER = 5

WORKDIR = Path.cwd()

DENY_LIST = [
    "rm -rf /", "sudo", "shutdown", "reboot",
    "mkfs", "dd if=", "> /dev/sda", "del", "rm", "rm -f", "chmod"
]

def check_deny_list(command: str) -> str | None:
    for pattern in DENY_LIST:
        if pattern in command:
            return f"Blocked: '{pattern}' is on the deny list"
    return None

PERMISSION_RULES = [
    {
        "tools": ["write_file", "edit_file"],
        "check": lambda args: not (WORKDIR / args.get("path", "")).resolve().is_relative_to(WORKDIR),
        "message": "Writing outside workspace",
    },
    {
        "tools": ["bash"],
        "check": lambda args: any(kw in args.get("command", "") for kw in ["rm ", "> /etc/", "chmod 777"]),
        "message": "Potentially destructive command",
    },
]

def check_rules(tool_name: str, args: dict) -> str | None:
    for rule in PERMISSION_RULES:
        if tool_name in rule["tools"] and rule["check"](args):
            return rule["message"]
    return None

def ask_user(tool_name: str, args: dict, reason: str) -> str:
    print(f"\n⚠  {reason}")
    print(f"   Tool: {tool_name}({args})")
    choice = input("   Allow? [y/N] ").strip().lower()
    return "allow" if choice in ("y", "yes") else "deny"

def check_permission(block) -> bool:
    if block.get("tool") == "bash":
        reason = check_deny_list(block.get("command", ""))
        if reason:
            print(f"\n⛔ {reason}")
            return False

    reason = check_rules(block.get("tool"), block.get("input", {}))
    if reason:
        decision = ask_user(block.get("tool"), block.get("input", {}), reason)
        if decision == "deny":
            return False

    return True

def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path


def messages_reducer(left : list, right : list) -> list:
    return left + right

class State(TypedDict):
    messages : Annotated[list[Dict[str, str]], messages_reducer]
    max_iterations : int



In [5]:

@tool
def run_bash(command : str) -> str:
    """
    Run a bash command and return the output.
    """
    dangerous_commands = ["rm", "sudo", "shutdown", "reboot"]
    if any(dangerous_command in command for dangerous_command in dangerous_commands):
        return "Error: Command contains potentially dangerous operations. Aborting execution."
    try:
        result = subprocess.run(command, shell=True, cwd=os.getcwd(), text=True, capture_output=True, timeout=120)
        out = (result.stdout + result.stderr).strip()
        return out[:10000] if out else "(No output)"
    except subprocess.CalledProcessError as e:
        return f"Error: {e.stderr}"

@tool 
def run_read(path, limit=None):
    """
    Read the contents of a file.
    """
    lines = safe_path(path).read_text().splitlines()
    if limit:
        lines = lines[:limit]
    return "\n".join(lines)

@tool
def run_write(path, content):
    """
    Write content to a file.
    """
    safe_path(path).write_text(content)
    return f"Wrote {len(content)} bytes to {path}"

@tool
def run_edit(path, old_text, new_text):
    """
    Edit a file by replacing the first occurrence of old_text with new_text.
    """
    text = safe_path(path).read_text()
    if old_text not in text:
        return "Error: text not found"
    safe_path(path).write_text(text.replace(old_text, new_text, 1))
    return f"Edited {path}"

@tool
def run_glob(pattern):
    """
    Glob for files matching a pattern.
    """
    import glob as g
    return "\n".join(g.glob(pattern, root_dir=WORKDIR))

In [6]:
def call_function(state: State) -> Command:
    all_messages = state.get("messages", [])
    #print("All Messages:", all_messages)
    if state.get("max_iterations", 0) <= 0 or not all_messages:
        if not all_messages:
            mess = f"No messages to process. Stopping execution."
        else:
            mess = f"Reached maximum iterations. Stopping execution."
        return Command(
            update={"messages": [AIMessage(content=mess)]},
            goto=END
        )
    
    last_message = all_messages[-1]

    content = last_message.content if hasattr(last_message, "content") else last_message.get("content", "")

    if "stop" in content.strip().lower():
        return Command(
            update={"messages": [AIMessage(content="Received STOP signal. Ending execution.")]},
            goto=END
        )
    elif "stop" not in content.strip().lower() and state.get("max_iterations", 0) > 0:        
        model = ChatOllama(model="gemma4", temperature=0.1)
        agent = create_agent(model=model, system_prompt=SYSTEM, tools=[run_bash, run_read, run_write, run_edit, run_glob])
        result = agent.invoke(state)
        #print(result)
        return Command(
            update={
                "messages": result.get("messages", []),
                "max_iterations": state.get("max_iterations", 0) - 1
            },
            goto="call_function"
        )
    else:
        return Command(
            goto=END
        )
  
builder = StateGraph(State)
builder.add_node("call_function", call_function)
builder.add_edge(START, "call_function")

# builder.add_edge(START, "call_function")
# builder.add_edge("call_function", END)

graph = builder.compile()

initial_state = {
    "messages": [
        {"role": "user", "content": 'Create a file called test.py that prints "hello", then read it back'}
    ],
    "max_iterations": MAX_ITER
}

result2 = graph.invoke(initial_state)
print("Final State:", result2)

Final State: {'messages': [{'role': 'user', 'content': 'Create a file called test.py that prints "hello", then read it back'}, HumanMessage(content='Create a file called test.py that prints "hello", then read it back', additional_kwargs={}, response_metadata={}, id='7e146adb-c3a0-4823-834b-2aa34f016170'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-05-24T18:58:50.4156832Z', 'done': True, 'done_reason': 'stop', 'total_duration': 118978245800, 'load_duration': 23351715400, 'prompt_eval_count': 426, 'prompt_eval_duration': 63874716300, 'eval_count': 128, 'eval_duration': 30864942200, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--019e5b58-dedd-7611-ba6b-794109388963-0', tool_calls=[{'name': 'run_write', 'args': {'content': 'print("hello")', 'path': 'test.py'}, 'id': '1bbce626-b6ca-4ddf-87f7-396e01cc5ff9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 426, 'output_t

In [ ]:
import os
from typing import Dict, Annotated, Literal, TypedDict
from pathlib import Path
import subprocess
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt
from langchain_ollama import ChatOllama
from langgraph.prebuilt import ToolNode

# --- Configuration & Security Rules ---
MAX_ITER = 5

WORKDIR = Path.cwd()
DENY_LIST = [
    "rm -rf /", "sudo", "shutdown", "reboot",
    "mkfs", "dd if=", "> /dev/sda", "del", "rm", "rm -f", "chmod"
]

def check_deny_list(command: str) -> str | None:
    for pattern in DENY_LIST:
        if pattern in command:
            return f"Blocked: '{pattern}' is on the deny list"
    return None

PERMISSION_RULES = [
    {
        "tools": ["run_write", "run_edit"],
        "check": lambda args: not (WORKDIR / args.get("path", "")).resolve().is_relative_to(WORKDIR),
        "message": "Writing outside workspace",
    },
    {
        "tools": ["run_bash"],
        "check": lambda args: any(kw in args.get("command", "") for kw in ["rm ", "> /etc/", "chmod 777"]),
        "message": "Potentially destructive command",
    },
]

def check_rules(tool_name: str, args: dict) -> str | None:
    for rule in PERMISSION_RULES:
        if tool_name in rule["tools"] and rule["check"](args):
            return rule["message"]
    return None

# --- Managed Tools ---
@tool
def run_bash(command: str) -> str:
    """Run a bash command and return the output."""
    try:
        result = subprocess.run(command, shell=True, cwd=os.getcwd(), text=True, capture_output=True, timeout=120)
        out = (result.stdout + result.stderr).strip()
        return out[:10000] if out else "(No output)"
    except Exception as e:
        return f"Error: {str(e)}"

@tool 
def run_read(path: str, limit: int = None) -> str:
    """Read the contents of a file."""
    # safe_path logic wrapped natively
    resolved = (WORKDIR / path).resolve()
    if not resolved.is_relative_to(WORKDIR):
        return "Error: Path escapes workspace"
    lines = resolved.read_text().splitlines()
    if limit:
        lines = lines[:limit]
    return "\n".join(lines)

@tool
def run_write(path: str, content: str) -> str:
    """Write content to a file."""
    resolved = (WORKDIR / path).resolve()
    if not resolved.is_relative_to(WORKDIR):
        return "Error: Path escapes workspace"
    resolved.write_text(content)
    return f"Wrote {len(content)} bytes to {path}"

@tool
def run_edit(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing the first occurrence of old_text with new_text."""
    resolved = (WORKDIR / path).resolve()
    if not resolved.is_relative_to(WORKDIR):
        return "Error: Path escapes workspace"
    text = resolved.read_text()
    if old_text not in text:
        return "Error: text not found"
    resolved.write_text(text.replace(old_text, new_text, 1))
    return f"Edited {path}"

TOOL_MAP = {
    "run_bash": run_bash, 
    "run_read": run_read, 
    "run_write": run_write, 
    "run_edit": run_edit
}
tools = list(TOOL_MAP.values())
tool_node = ToolNode(tools)

# --- Graph State Schema ---
class State(TypedDict):
    messages: Annotated[list, add_messages]
    max_iterations: int

# --- Graph Nodes ---
def call_model(state: State) -> Dict:
    """Node responsible solely for querying the LLM."""
    if state.get("max_iterations", 0) <= 0:
        return {"messages": [AIMessage(content="Reached maximum iterations. Stopping.")]}
        
    model = ChatOllama(model="gemma4", temperature=0.1)
    # Bind tools natively so the model formats valid Tool Calls out-of-the-box
    model_with_tools = model.bind_tools(tools)
    
    response = model_with_tools.invoke(state["messages"])
    return {
        "messages": [response],
        "max_iterations": state.get("max_iterations", 0) - 1
    }

def verify_and_route(state: State) -> Command[Literal["tools", "call_model", "__end__"]]:
    """Graph node executing validation logic before routing."""
    last_message = state["messages"][-1]
    
    # If the model didn't call any tools or requested to stop, exit
    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return Command(goto=END)
        
    if "stop" in last_message.content.lower():
        return Command(goto=END)

    # Evaluate tools deterministically
    for tool_call in last_message.tool_calls:
        t_name = tool_call["name"]
        t_args = tool_call["args"]
        
        # 1. Check Hard Deny Lists
        if t_name == "run_bash":
            deny_reason = check_deny_list(t_args.get("command", ""))
            if deny_reason:
                return Command(
                    goto="call_model",
                    update={"messages": [ToolMessage(content=f"⛔ {deny_reason}", tool_call_id=tool_call["id"])]}
                )
                
        # 2. Check Conditional Security Rules
        rule_reason = check_rules(t_name, t_args)
        if rule_reason:
            human_response = interrupt({
                "reason": rule_reason,
                "tool": t_name,
                "args": t_args
            })
            
            if human_response.get("action") != "allow":
                return Command(
                    goto="call_model",
                    update={"messages": [ToolMessage(content="Permission denied by user.", tool_call_id=tool_call["id"])]}
                )

    # If everything looks safe, route to the tool execution node
    return Command(goto="tools")

# --- Building the Graph Architecture ---
builder = StateGraph(State)

builder.add_node("call_model", call_model)
builder.add_node("verify_node", verify_and_route) # <-- Make it a node!
builder.add_node("tools", tool_node)

builder.add_edge(START, "call_model")
builder.add_edge("call_model", "verify_node")     # <-- Direct edge instead of conditional
builder.add_edge("tools", "call_model")

graph = builder.compile(checkpointer=MemorySaver())

initial_state = {
    "messages": [
        {"role": "user", "content": 'Delete the test.py file if it exists'}
    ],
    "max_iterations": MAX_ITER
}

config = {"configurable": {"thread_id": "session_123"}}


result2 = graph.invoke(initial_state, config=config)
print("Final State:", result2)

KeyboardInterrupt: 